In [248]:
from MLP import MLP
from Engine import MicroTensor
from Optimizers import Adam
from utils import create_Batches

In [249]:
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from copy import deepcopy

In [250]:
mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')

In [251]:
mnist["data"].shape

(70000, 784)

In [252]:
mnist["target"].shape

(70000,)

In [253]:
X = mnist["data"]
Y = mnist["target"]

In [254]:
x, x_test, y, y_test = train_test_split(X, Y, test_size=0.2, stratify=Y)

In [255]:
x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.2, stratify=y)

In [256]:
x_train.shape

(44800, 784)

In [257]:
y_train.shape

(44800,)

In [258]:
np.unique(y_train)

array(['0', '1', '2', '3', '4', '5', '6', '7', '8', '9'], dtype=object)

In [259]:
def vectorize_y(y):
    out = []
    for value in y:
        vec = np.zeros((10, ))
        vec[int(value)] = 1
        out.append(vec.copy())
    return np.array(out)

In [260]:
y_train = vectorize_y(y_train)
y_val = vectorize_y(y_val)
y_test = vectorize_y(y_test)

In [261]:
x_train = x_train[..., None]
x_val = x_val[..., None]
x_test = x_test[..., None]

y_train = y_train[..., None]
y_val = y_val[..., None]
y_test = y_test[..., None]

In [262]:
optimizer = Adam(lr=0.09)

In [263]:
mnist_model = MLP(
    components=[
        ["Linear", 784, 512],
        ["Tanh"],
        ["Linear", 512, 10],
    ],
    LossFunction="CrossEntropy",
    optimizer=optimizer
)

[['Linear', 784, 512], ['Tanh'], ['Linear', 512, 10]]
in : 784, out : 512
in : 512, out : 10


In [264]:
BatchSize = 50

In [265]:
x_train_batches, y_train_batches = create_Batches(x_train, y_train, bacthSize=BatchSize)

In [266]:
epochs = 12

In [267]:
x_val_MicroTensor = MicroTensor(x_val)
x_train_MicroTensor = MicroTensor(x_train)

In [268]:
def accuracy(y_true, y_pred):
    y_true1 = y_true.reshape(y_true.shape[0], -1)
    y_pred1 = y_pred.reshape(y_true.shape[0], -1)
    
    return (y_true1.argmax(axis=1) == y_pred1.argmax(axis=1)).sum()/len(y_true1)

In [269]:
best_val_accuracy = -np.inf
best_weights = None
for i in range(epochs):
    mnist_model.training = True
    for x_batch, y_batch in zip(x_train_batches, y_train_batches):
        x_microTensor = MicroTensor(x_batch)
        topo = mnist_model(x_microTensor, y_batch)
        topo.backward()
        mnist_model.update()
        
        mnist_model.zero_grad()
    
    mnist_model.training = False
    val_accuracy = accuracy(y_val, mnist_model(x_val_MicroTensor))
    if val_accuracy > best_val_accuracy:
        best_weights = deepcopy(mnist_model.steps)
        best_val_accuracy = val_accuracy
    print(f"epoch {i}, train accuracy : {accuracy(y_train, mnist_model(x_train_MicroTensor))}, val accuracy : {val_accuracy}")

c:\Users\Amine\Desktop\python workspace\MicroTensorGrad\Engine.py:72: RuntimeWarning: divide by zero encountered in log
  out = MicroTensor(np.mean(-np.sum(Y*np.log(self.Softmax()), axis=1), axis=0))
c:\Users\Amine\Desktop\python workspace\MicroTensorGrad\Engine.py:72: RuntimeWarning: invalid value encountered in multiply
  out = MicroTensor(np.mean(-np.sum(Y*np.log(self.Softmax()), axis=1), axis=0))


epoch 0, train accuracy : 0.8465178571428571, val accuracy : 0.8433035714285714
epoch 1, train accuracy : 0.8688392857142857, val accuracy : 0.8633035714285714
epoch 2, train accuracy : 0.92171875, val accuracy : 0.9164285714285715
epoch 3, train accuracy : 0.9313169642857143, val accuracy : 0.9255357142857142
epoch 4, train accuracy : 0.9491964285714286, val accuracy : 0.9391964285714286
epoch 5, train accuracy : 0.94171875, val accuracy : 0.9363392857142857
epoch 6, train accuracy : 0.9477901785714286, val accuracy : 0.9433035714285715
epoch 7, train accuracy : 0.9509375, val accuracy : 0.9434821428571428
epoch 8, train accuracy : 0.9402901785714286, val accuracy : 0.9351785714285714
epoch 9, train accuracy : 0.9578794642857142, val accuracy : 0.9509821428571429
epoch 10, train accuracy : 0.96390625, val accuracy : 0.9569642857142857
epoch 11, train accuracy : 0.9462723214285714, val accuracy : 0.9380357142857143


In [270]:
mnist_model.steps = best_weights

In [271]:
mnist_model.training = False

In [272]:
accuracy(y_test, mnist_model(MicroTensor(x_test)))

np.float64(0.9544285714285714)